In [1]:
# 04 - Profile clean data

# Purpose: check that the saved clean data (`data/bank_marketing_clean.parquet`) matches what `docs/decision-log.md` says was done, and nothing more. Each check is reconciled against the raw data.

In [2]:
import pandas as pd

RAW_PATH = "../data/bank-additional-full.csv"
CLEAN_PATH = "../data/bank_marketing_clean.parquet"

raw_df = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
clean_df = pd.read_parquet(CLEAN_PATH)

print(raw_df.shape, clean_df.shape)

(41188, 21) (41176, 23)


In [3]:
# Check 1: structure, missing values, duplicates, and "unknown" counts
print(f"Shape: {clean_df.shape}")
print(f"Duplicate rows: {clean_df.duplicated().sum()}")
print()

missing_per_column = clean_df.isna().sum()
print("Missing values (only columns that have any):")
print(missing_per_column[missing_per_column > 0])
print()

# The first 21 clean columns are the raw columns in the same order, so we can pair the names
raw_to_clean_names = dict(zip(raw_df.columns, clean_df.columns[:21]))
raw_renamed_df = raw_df.rename(columns=raw_to_clean_names)
removed_rows_df = raw_renamed_df[raw_df.duplicated()]

unknown_columns = [
    "credit_default", "education", "housing_loan",
    "personal_loan", "job", "marital_status",
]
unknown_check = pd.DataFrame({
    "raw_unknown": (raw_renamed_df[unknown_columns] == "unknown").sum(),
    "clean_unknown": (clean_df[unknown_columns] == "unknown").sum(),
    "in_removed_rows": (removed_rows_df[unknown_columns] == "unknown").sum(),
})
unknown_check["difference_explained"] = (
    unknown_check["raw_unknown"] - unknown_check["clean_unknown"]
    == unknown_check["in_removed_rows"]
)
unknown_check

Shape: (41176, 23)
Duplicate rows: 0

Missing values (only columns that have any):
days_since_previous_contact    39661
dtype: int64



,raw_unknown,clean_unknown,in_removed_rows,difference_explained
credit_default,8597,8596,1,True
education,1731,1730,1,True
housing_loan,990,990,0,True
personal_loan,990,990,0,True
job,330,330,0,True
marital_status,80,80,0,True


In [4]:
# Check 2: category reconciliation, y balance, nonexistent vs previous = 0
categorical_columns = [
    "job", "marital_status", "education", "credit_default", "housing_loan",
    "personal_loan", "contact_type", "contact_month", "contact_weekday",
    "previous_campaign_outcome", "subscribed_term_deposit",
]

mismatches = []
for column in categorical_columns:
    counts = pd.DataFrame({
        "raw": raw_renamed_df[column].value_counts(),
        "clean": clean_df[column].value_counts(),
        "removed": removed_rows_df[column].value_counts(),
    }).fillna(0)
    all_explained = (counts["raw"] - counts["clean"] == counts["removed"]).all()
    print(f"{column}: {len(counts)} categories, all reconcile: {all_explained}")
    if not all_explained:
        mismatches.append(column)
print()
print("Columns with a mismatch:", mismatches)
print()

# y balance on the clean data
y_counts = clean_df["subscribed_term_deposit"].value_counts()
print(y_counts)
print(f"Total: {y_counts.sum()}")
print(f"Yes rate: {(clean_df['subscribed_term_deposit'] == 'yes').mean() * 100:.3f}%")
print()

# nonexistent vs previous = 0
print("previous_campaign_outcome = nonexistent vs contacts_before_campaign = 0:")
print(pd.crosstab(
    clean_df["previous_campaign_outcome"] == "nonexistent",
    clean_df["contacts_before_campaign"] == 0,
    rownames=["outcome is nonexistent"],
    colnames=["contacts before is 0"],
))

job: 12 categories, all reconcile: True
marital_status: 4 categories, all reconcile: True
education: 8 categories, all reconcile: True
credit_default: 3 categories, all reconcile: True
housing_loan: 3 categories, all reconcile: True
personal_loan: 3 categories, all reconcile: True
contact_type: 2 categories, all reconcile: True
contact_month: 10 categories, all reconcile: True
contact_weekday: 5 categories, all reconcile: True
previous_campaign_outcome: 3 categories, all reconcile: True
subscribed_term_deposit: 2 categories, all reconcile: True

Columns with a mismatch: []

subscribed_term_deposit
no     36537
yes     4639
Name: count, dtype: int64
Total: 41176
Yes rate: 11.266%

previous_campaign_outcome = nonexistent vs contacts_before_campaign = 0:
contacts before is 0    False  True 
outcome is nonexistent              
False                    5625      0
True                        0  35551


In [5]:
# Check 3: the inferred campaign year, rebuilt from the raw data
month_numbers = {"mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8,
                 "sep": 9, "oct": 10, "nov": 11, "dec": 12}

raw_month = raw_renamed_df["contact_month"].map(month_numbers)
raw_year = 2008 + (raw_month < raw_month.shift()).cumsum()
removed_by_year = raw_year[raw_df.duplicated()].value_counts().sort_index()

year_check = pd.DataFrame({
    "raw_customers": raw_year.value_counts().sort_index(),
    "removed_duplicates": removed_by_year,
    "clean_customers": clean_df["inferred_campaign_year"].value_counts().sort_index(),
}).fillna(0).astype(int)
year_check["difference_explained"] = (
    year_check["raw_customers"] - year_check["removed_duplicates"] == year_check["clean_customers"]
)
print(year_check)
print()

clean_month = clean_df["contact_month"].map(month_numbers)
print(f"Column type: {clean_df['inferred_campaign_year'].dtype}, "
      f"values: {sorted(clean_df['inferred_campaign_year'].unique().tolist())}")
print(f"Year never goes down along the file: {clean_df['inferred_campaign_year'].is_monotonic_increasing}")
print(f"Month never goes down within a year: "
      f"{(clean_month.groupby(clean_df['inferred_campaign_year']).diff().dropna() >= 0).all()}")
print(f"Missing values in the new column: {clean_df['inferred_campaign_year'].isna().sum()}")
print()

clean_subscribed = clean_df["subscribed_term_deposit"] == "yes"
print("Yes rate by year (%):")
print((clean_subscribed.groupby(clean_df["inferred_campaign_year"]).mean() * 100).round(1))

      raw_customers  removed_duplicates  clean_customers  difference_explained
2008          27690                   8            27682                  True
2009          11440                   4            11436                  True
2010           2058                   0             2058                  True

Column type: int64, values: [2008, 2009, 2010]
Year never goes down along the file: True
Month never goes down within a year: True
Missing values in the new column: 0

Yes rate by year (%):
inferred_campaign_year
2008     4.8
2009    19.5
2010    52.1
Name: subscribed_term_deposit, dtype: float64
